---
# Section from: explore_data.ipynb
---

# Tortilla Prices in Mexico: Preliminary Exploration

This notebook explores the dataset containing historical tortilla prices across different states and store types in Mexico.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid")

# Load the dataset
df = pd.read_csv('data/tortilla_prices.csv')
df.head()

## Data Cleaning
We need to handle any missing values in the dataset. Since `Price per kilogram` is our primary target variable, we will drop rows where this value is NaN.

In [ ]:
# Check for missing values
print("Missing values before cleaning:")
print(df.isna().sum())

# Drop rows with NaN in 'Price per kilogram'
df = df.dropna(subset=['Price per kilogram'])

print("
Missing values after cleaning:")
print(df.isna().sum())

print(f"
Total remaining rows: {len(df)}")

In [ ]:
df.info()
df.describe()

## Price Trends Over Time

In [ ]:
# Create a proper datetime column
df['Date'] = pd.to_datetime(df[['Year', 'Month', 'Day']])

plt.figure(figsize=(12, 6))
sns.lineplot(data=df, x='Year', y='Price per kilogram', hue='Store type', errorbar=None)
plt.title('Average Tortilla Price per kg Over Time (by Store Type)')
plt.ylabel('Price (MXN)')
plt.show()

## Price Distribution by Store Type

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(data=df, x='Store type', y='Price per kilogram', palette='Set2')
plt.title('Price Distribution by Store Type')
plt.ylabel('Price (MXN)')
plt.show()

## Average Price by State (Most Recent Year)

In [ ]:
recent_year = df['Year'].max()
recent_df = df[df['Year'] == recent_year]

state_prices = recent_df.groupby('State')['Price per kilogram'].mean().sort_values(ascending=False).reset_index()

plt.figure(figsize=(14, 8))
sns.barplot(data=state_prices, x='Price per kilogram', y='State', palette='viridis')
plt.title(f'Average Tortilla Price per kg by State in {recent_year}')
plt.xlabel('Average Price (MXN)')
plt.ylabel('State')
plt.show()

---
# Section from: arima_forecast.ipynb
---

# ARIMA Statistical Study: Tortilla Prices Forecasting

In this notebook, we fit an Auto-ARIMA model to the historical national average tortilla prices in Mexico to forecast future prices.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
import pmdarima as pm
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")

## 1. Data Preprocessing
We aggregate the dataset by year and month to create a single univariate time series representing the national average price.

In [ ]:
df = pd.read_csv('data/tortilla_prices.csv')
df = df.dropna(subset=['Price per kilogram'])

monthly_price = df.groupby(['Year', 'Month'])['Price per kilogram'].mean().reset_index()
monthly_price['Date'] = pd.to_datetime(monthly_price[['Year', 'Month']].assign(DAY=1))
monthly_price = monthly_price.set_index('Date').sort_index()

ts = monthly_price['Price per kilogram']
ts.plot(figsize=(10, 5), title='National Average Tortilla Price')
plt.show()

## 2. Stationarity Check (Augmented Dickey-Fuller Test)

In [ ]:
adf_result = adfuller(ts)
print(f'ADF Statistic: {adf_result[0]:.4f}')
print(f'p-value: {adf_result[1]:.4f}')
if adf_result[1] > 0.05:
    print("The series is not stationary. Differencing is required.")
else:
    print("The series is stationary.")

## 3. Auto-ARIMA Model Selection & Fitting
We use `pmdarima` to automatically search for the optimal (p, d, q) parameters.

In [ ]:
auto_model = pm.auto_arima(ts, start_p=1, start_q=1,
                           max_p=3, max_q=3, m=12,
                           start_P=0, seasonal=True,
                           d=1, D=1, trace=True,
                           error_action='ignore',  
                           suppress_warnings=True, 
                           stepwise=True)

print(auto_model.summary())

## 4. Forecasting Future Prices (24 Months)

In [ ]:
forecast_steps = 24
forecast, conf_int = auto_model.predict(n_periods=forecast_steps, return_conf_int=True)

forecast_index = pd.date_range(start=ts.index[-1] + pd.DateOffset(months=1), periods=forecast_steps, freq='MS')
forecast_series = pd.Series(forecast.values, index=forecast_index)
lower_series = pd.Series(conf_int[:, 0], index=forecast_index)
upper_series = pd.Series(conf_int[:, 1], index=forecast_index)

plt.figure(figsize=(12, 6))
plt.plot(ts, label='Historical Average Price', color='blue')
plt.plot(forecast_series, label='ARIMA Forecast', color='red')
plt.fill_between(forecast_index, lower_series, upper_series, color='red', alpha=0.2, label='95% Confidence Interval')

plt.title('National Average Tortilla Price: Historical & 2-Year Forecast', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Price per kg (MXN)', fontsize=12)
plt.legend(loc='upper left')
plt.show()

---
# Section from: wage_vs_tortilla.ipynb
---

# Tortilla Prices vs. Minimum Wage Growth in Mexico

In this notebook, we compare the historical growth of the national average tortilla price against the growth of the general minimum wage (Salario Mínimo General) in Mexico from 2007 to 2024.

Both metrics are normalized to a base index of 100 in 2007 to clearly accentuate the difference in their relative growth rates.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")

## 1. Load and Prepare Tortilla Prices

In [ ]:
df = pd.read_csv('data/tortilla_prices.csv')
df = df.dropna(subset=['Price per kilogram'])

# Aggregate average national price by year
yearly_price = df.groupby('Year')['Price per kilogram'].mean().reset_index()
yearly_price.head()

## 2. Introduce Historical Minimum Wage Data
The data below is sourced from the official decrees by CONASAMI (Comisión Nacional de los Salarios Mínimos).

In [ ]:
# Historical Minimum Wage in Mexico (2007-2024) in Pesos per day
wage_data = {
    'Year': [2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024],
    'Minimum Wage (MXN/day)': [50.57, 52.59, 54.80, 57.46, 59.82, 62.33, 64.76, 67.29, 70.10, 73.04, 80.04, 88.36, 102.68, 123.22, 141.70, 172.87, 207.44, 248.93]
}
df_wage = pd.DataFrame(wage_data)
df_wage.head()

## 3. Merge and Calculate Growth Index
We calculate the index by dividing the value of any given year by the base value in 2007, multiplied by 100.

In [ ]:
merged = pd.merge(yearly_price, df_wage, on='Year', how='inner')

# Base Year values
base_price = merged.loc[merged['Year'] == 2007, 'Price per kilogram'].values[0]
base_wage = merged.loc[merged['Year'] == 2007, 'Minimum Wage (MXN/day)'].values[0]

merged['Price Growth Index'] = (merged['Price per kilogram'] / base_price) * 100
merged['Wage Growth Index'] = (merged['Minimum Wage (MXN/day)'] / base_wage) * 100

merged[['Year', 'Price Growth Index', 'Wage Growth Index']].tail()

## 4. Visualization

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(merged['Year'], merged['Price Growth Index'], marker='o', label='Tortilla Price Growth', color='red', linewidth=2)
plt.plot(merged['Year'], merged['Wage Growth Index'], marker='s', label='Minimum Wage Growth', color='green', linewidth=2)

plt.axhline(100, color='grey', linestyle='--', alpha=0.6)

plt.title('Growth Comparison: Tortilla Prices vs Minimum Wage in Mexico (2007 = 100)', fontsize=16)
plt.xlabel('Year', fontsize=12)
plt.ylabel('Growth Index (2007 = 100)', fontsize=12)
plt.xticks(merged['Year'], rotation=45)
plt.legend(loc='upper left', fontsize=12)

# Annotate final values to accentuate the difference
final_year = merged.iloc[-1]
plt.annotate(f"{final_year['Price Growth Index']:.1f}%", 
             (final_year['Year'], final_year['Price Growth Index']), 
             textcoords="offset points", xytext=(0,10), ha='center', color='red', fontweight='bold')

plt.annotate(f"{final_year['Wage Growth Index']:.1f}%", 
             (final_year['Year'], final_year['Wage Growth Index']), 
             textcoords="offset points", xytext=(0,-15), ha='center', color='green', fontweight='bold')

plt.show()